### 1. CONFIGURATION AND ENVIRONMENT

In [ ]:
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
from tqdm import tqdm
import time
import random
import math

# -------------------------------
# Set Random Seed for Reproducibility
# -------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# -------------------------------
# Device Configuration
# -------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -------------------------------
# Data Paths
# -------------------------------
FEATURES_DIR = '/kaggle/input/msegcn-individual/MSE_GCN_features_individual'
JSON_PATH = '/kaggle/input/json-files/nslt_300.json'

# -------------------------------
# Hyperparameter Configuration
# -------------------------------
CONFIG = {
    'SEQ_LEN': 64,
    'BATCH_SIZE': 32,
    'EPOCHS': 300,
    'BASE_CHANNELS': 24,  # Reduced for lightweight model
    'DROPOUT': 0.25,
    'LEARNING_RATE': 0.001,
    'WEIGHT_DECAY': 0.05,
    'LABEL_SMOOTHING': 0.1,
    'MIXUP_ALPHA': 0.2,
    'MIXUP_PROB': 0.5,
    'WARMUP_EPOCHS': 10,
    'MIN_LR': 1e-6,
    'GRAD_CLIP': 1.0,
    'PATIENCE': 60,
}


### 1. GRAPH


In [ ]:

class Graph:
    def __init__(self):
        self.num_node = 65
        self.self_link = [(i, i) for i in range(self.num_node)]
        
        self.inward_bone_link = [
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
            (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
            (12, 11), (10, 8), (8, 6), (6, 5), (5, 4), (4, 0),
            (9, 7), (7, 3), (3, 2), (2, 1), (1, 0),
            (15, 23), (16, 44)
        ]
        
        for hand_start_idx in [23, 44]:
            self.inward_bone_link.extend([
                (hand_start_idx, hand_start_idx + 1), 
                (hand_start_idx + 1, hand_start_idx + 2),
                (hand_start_idx + 2, hand_start_idx + 3), 
                (hand_start_idx + 3, hand_start_idx + 4)
            ])
            for finger_start in range(5, 21, 4):
                self.inward_bone_link.extend([
                    (hand_start_idx, hand_start_idx + finger_start),
                    (hand_start_idx + finger_start, hand_start_idx + finger_start + 1),
                    (hand_start_idx + finger_start + 1, hand_start_idx + finger_start + 2),
                    (hand_start_idx + finger_start + 2, hand_start_idx + finger_start + 3)
                ])
        
        self.edge = self.self_link + self.inward_bone_link
        self.A = torch.zeros(self.num_node, self.num_node)
        for i, j in self.edge:
            self.A[j, i] = 1
            self.A[i, j] = 1



### Model Archetecture

In [ ]:



# ============================================================================
# 2. LIGHTWEIGHT BUILDING BLOCKS
# ============================================================================
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=1, stride=1, padding=0):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size, stride, padding, groups=in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU()
    
    def forward(self, x):
        return self.act(self.bn(self.pointwise(self.depthwise(x))))


class SimpleAttention(nn.Module):
    """Lightweight attention - just 2 conv layers"""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels // 4, 1),
            nn.SiLU(),
            nn.Conv2d(channels // 4, channels, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        attn = self.conv(x.mean(dim=2, keepdim=True))
        return x * attn


class AdaptiveGCN(nn.Module):
    def __init__(self, in_ch, out_ch, A):
        super().__init__()
        # REDUCED from 3 subsets to 2
        self.A = nn.Parameter(torch.stack([A, A]), requires_grad=True)
        self.conv1 = DepthwiseSeparableConv(in_ch, out_ch, 1)
        self.conv2 = DepthwiseSeparableConv(in_ch, out_ch, 1)
        
    def forward(self, x):
        N, C, T, V = x.shape
        x_reshaped = x.view(N, C * T, V)
        
        z1 = torch.matmul(x_reshaped, self.A[0]).view(N, C, T, V)
        z1 = self.conv1(z1)
        
        z2 = torch.matmul(x_reshaped, self.A[1]).view(N, C, T, V)
        z2 = self.conv2(z2)
        
        return z1 + z2


class LightweightSTGCNBlock(nn.Module):
    """Simplified block - no heavy transformers"""
    def __init__(self, in_ch, out_ch, A, stride=1, use_attention=False, dropout=0.25):
        super().__init__()
        self.gcn = AdaptiveGCN(in_ch, out_ch, A)
        self.tcn = DepthwiseSeparableConv(
            out_ch, out_ch, kernel_size=(3, 1), stride=(stride, 1), padding=(1, 0)
        )
        
        self.use_attention = use_attention
        if use_attention:
            self.attention = SimpleAttention(out_ch)
        
        self.dropout = nn.Dropout2d(dropout)
        
        self.residual = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=(stride, 1)),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()
        
    def forward(self, x):
        res = self.residual(x)
        x = self.gcn(x)
        x = self.dropout(x)
        x = self.tcn(x)
        
        if self.use_attention:
            x = self.attention(x)
        
        return F.silu(x + res)


class StreamFusion(nn.Module):
    """Simplified fusion"""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1),
            nn.BatchNorm2d(channels),
            nn.SiLU()
        )
    
    def forward(self, x1, x2):
        return self.conv(torch.cat([x1, x2], dim=1))


# ============================================================================
# 3. LIGHTWEIGHT MAIN MODEL (600K params)
# ============================================================================
class LightweightTTSTGCN(nn.Module):
    """Ultra-lightweight model - 600K parameters"""
    def __init__(self, num_classes, base_ch=24, dropout=0.25):
        super().__init__()
        
        graph = Graph()
        A = graph.A
        
        self.bn_joint = nn.BatchNorm2d(4)
        self.bn_bone = nn.BatchNorm2d(2)
        
        # Joint stream - SIMPLE
        self.joint_stream = nn.Sequential(
            LightweightSTGCNBlock(4, base_ch, A, use_attention=False, dropout=dropout),
            LightweightSTGCNBlock(base_ch, base_ch * 2, A, use_attention=True, dropout=dropout),
            LightweightSTGCNBlock(base_ch * 2, base_ch * 3, A, stride=2, use_attention=True, dropout=dropout),
        )
        
        # Bone stream - SIMPLE
        self.bone_stream = nn.Sequential(
            LightweightSTGCNBlock(2, base_ch, A, use_attention=False, dropout=dropout),
            LightweightSTGCNBlock(base_ch, base_ch * 2, A, use_attention=True, dropout=dropout),
            LightweightSTGCNBlock(base_ch * 2, base_ch * 3, A, stride=2, use_attention=True, dropout=dropout),
        )
        
        # Fusion
        self.fusion = StreamFusion(base_ch * 3)
        
        # Post-fusion
        self.post_fusion = nn.Sequential(
            LightweightSTGCNBlock(base_ch * 3, base_ch * 4, A, stride=2, use_attention=True, dropout=dropout),
            LightweightSTGCNBlock(base_ch * 4, base_ch * 5, A, use_attention=True, dropout=dropout),
        )
        
        # Classifier
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_ch * 5, num_classes)
        
    def forward(self, x_joint, x_bone):
        x_joint = self.bn_joint(x_joint)
        x_bone = self.bn_bone(x_bone)
        
        f_joint = self.joint_stream(x_joint)
        f_bone = self.bone_stream(x_bone)
        
        f_fused = self.fusion(f_joint, f_bone)
        out = self.post_fusion(f_fused)
        
        out = self.pool(out).view(out.size(0), -1)
        out = self.dropout(out)
        out = self.fc(out)
        
        return out




Using device: cuda


## Dataset and Preprocessing 

In [ ]:
# ============================================================================
# 4. AUGMENTATION
# ============================================================================
class ModerateAugmentation:
    def __init__(self):
        self.shear_range = 0.1
        self.rotate_range = 15
        self.scale_range = (0.9, 1.1)
        self.temporal_crop_ratio = 0.9
        self.spatial_drop_prob = 0.05
        self.temporal_mask_prob = 0.1
    
    def random_rotate(self, joints):
        angle = np.random.uniform(-self.rotate_range, self.rotate_range)
        angle_rad = np.deg2rad(angle)
        cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
        rot_matrix = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
        T, V, C = joints.shape
        joints_rot = joints.copy()
        joints_rot[:, :, :2] = np.dot(joints[:, :, :2].reshape(-1, 2), rot_matrix.T).reshape(T, V, 2)
        return joints_rot
    
    def random_scale(self, joints):
        return joints * np.random.uniform(*self.scale_range)
    
    def random_shear(self, joints):
        shear_x = np.random.uniform(-self.shear_range, self.shear_range)
        shear_y = np.random.uniform(-self.shear_range, self.shear_range)
        shear_matrix = np.array([[1, shear_x], [shear_y, 1]])
        T, V, C = joints.shape
        joints_shear = joints.copy()
        joints_shear[:, :, :2] = np.dot(joints[:, :, :2].reshape(-1, 2), shear_matrix.T).reshape(T, V, 2)
        return joints_shear
    
    def temporal_crop(self, joints, bones):
        T = joints.shape[0]
        new_T = int(T * self.temporal_crop_ratio)
        if new_T >= T:
            return joints, bones
        start = np.random.randint(0, T - new_T + 1)
        return joints[start:start+new_T], bones[start:start+new_T]
    
    def spatial_dropout(self, joints):
        T, V, C = joints.shape
        mask = np.random.rand(V) > self.spatial_drop_prob
        joints_dropped = joints.copy()
        joints_dropped[:, ~mask, :] = 0
        return joints_dropped
    
    def temporal_masking(self, joints):
        T = joints.shape[0]
        num_masks = int(T * self.temporal_mask_prob)
        if num_masks == 0:
            return joints
        joints_masked = joints.copy()
        for _ in range(num_masks):
            joints_masked[np.random.randint(0, T)] = 0
        return joints_masked
    
    def __call__(self, joints, bones, training=True):
        if not training:
            return joints, bones
        
        if np.random.rand() > 0.3:
            joints = self.random_rotate(joints)
        if np.random.rand() > 0.3:
            joints = self.random_scale(joints)
        if np.random.rand() > 0.5:
            joints = self.random_shear(joints)
        if np.random.rand() > 0.5:
            joints, bones = self.temporal_crop(joints, bones)
        if np.random.rand() > 0.5:
            joints = self.spatial_dropout(joints)
        if np.random.rand() > 0.5:
            joints = self.temporal_masking(joints)
        
        return joints, bones


# ============================================================================
# 5. DATASET
# ============================================================================
class EnhancedSignLanguageDataset(Dataset):
    def __init__(self, video_ids, labels_dict, features_dir, seq_len, 
                 augmentation=None, training=True):
        self.video_ids = video_ids
        self.labels = labels_dict
        self.features_dir = features_dir
        self.seq_len = seq_len
        self.augmentation = augmentation
        self.training = training
        self.bone_to_joint_map = {0: 13, 1: 15, 2: 14, 3: 16, 4: 11}
    
    def __len__(self):
        return len(self.video_ids)
    
    def __getitem__(self, idx):
        video_id = self.video_ids[idx]
        file_path = os.path.join(self.features_dir, f"{video_id}.npz")
        data = np.load(file_path)
        
        joint_seq = data['joint_sequence'].astype(np.float32)
        bone_seq_flat = data['bone_sequence'].astype(np.float32)
        
        num_frames = bone_seq_flat.shape[0]
        bone_seq = np.zeros((num_frames, 65, 2), dtype=np.float32)
        for bone_idx, joint_idx in self.bone_to_joint_map.items():
            start_col = bone_idx * 2
            bone_seq[:, joint_idx, :] = bone_seq_flat[:, start_col:start_col+2]
        
        if self.augmentation:
            joint_seq, bone_seq = self.augmentation(joint_seq, bone_seq, self.training)
        
        original_len = joint_seq.shape[0]
        if original_len < self.seq_len:
            pad_len = self.seq_len - original_len
            joint_seq = np.pad(joint_seq, ((0, pad_len), (0, 0), (0, 0)), 'constant')
            bone_seq = np.pad(bone_seq, ((0, pad_len), (0, 0), (0, 0)), 'constant')
        elif original_len > self.seq_len:
            if self.training:
                start = np.random.randint(0, original_len - self.seq_len + 1)
            else:
                start = (original_len - self.seq_len) // 2
            joint_seq = joint_seq[start:start+self.seq_len]
            bone_seq = bone_seq[start:start+self.seq_len]
        
        label = self.labels[video_id]
        joint_seq_tensor = torch.tensor(joint_seq).permute(2, 0, 1)
        bone_seq_tensor = torch.tensor(bone_seq).permute(2, 0, 1)
        
        return joint_seq_tensor, bone_seq_tensor, torch.tensor(label, dtype=torch.long)



### 6. TRAINING UTILITIES


In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    
    def forward(self, pred, target):
        n_classes = pred.size(-1)
        log_prob = F.log_softmax(pred, dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_prob)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_prob, dim=-1))


class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, max_epochs, min_lr=1e-6):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr = min_lr
        super().__init__(optimizer)
    
    def get_lr(self):
        if self.last_epoch < self.warmup_epochs:
            alpha = self.last_epoch / self.warmup_epochs
            return [base_lr * alpha for base_lr in self.base_lrs]
        else:
            progress = (self.last_epoch - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs)
            cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
            return [self.min_lr + (base_lr - self.min_lr) * cosine_decay 
                    for base_lr in self.base_lrs]


def mixup_data(x_joint, x_bone, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    batch_size = x_joint.size(0)
    index = torch.randperm(batch_size).to(x_joint.device)
    
    mixed_joint = lam * x_joint + (1 - lam) * x_joint[index]
    mixed_bone = lam * x_bone + (1 - lam) * x_bone[index]
    y_a, y_b = y, y[index]
    
    return mixed_joint, mixed_bone, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


class EarlyStopping:
    def __init__(self, patience=60, min_delta=0.3):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        
        if score > (self.best_score + self.min_delta):
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop


def train_one_epoch(model, loader, optimizer, criterion, device, config):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for joint_batch, bone_batch, labels in tqdm(loader, desc="Training", leave=False):
        joint_batch = joint_batch.to(device)
        bone_batch = bone_batch.to(device)
        labels = labels.to(device)
        
        use_mixup = (np.random.rand() < config['MIXUP_PROB'])
        
        if use_mixup:
            joint_batch, bone_batch, labels_a, labels_b, lam = mixup_data(
                joint_batch, bone_batch, labels, config['MIXUP_ALPHA']
            )
            optimizer.zero_grad()
            outputs = model(joint_batch, bone_batch)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            optimizer.zero_grad()
            outputs = model(joint_batch, bone_batch)
            loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config['GRAD_CLIP'])
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device, num_tta=3):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for joint_batch, bone_batch, labels in tqdm(loader, desc="Validating", leave=False):
            joint_batch = joint_batch.to(device)
            bone_batch = bone_batch.to(device)
            labels = labels.to(device)
            
            tta_preds = []
            for _ in range(num_tta):
                outputs = model(joint_batch, bone_batch)
                tta_preds.append(F.softmax(outputs, dim=1))
            
            outputs_avg = torch.stack(tta_preds).mean(0)
            loss = criterion(outputs_avg.log(), labels)
            
            running_loss += loss.item()
            all_preds.append(outputs_avg.cpu())
            all_labels.append(labels.cpu())
    
    all_preds = torch.cat(all_preds, 0)
    all_labels = torch.cat(all_labels, 0)
    
    _, predicted = all_preds.max(1)
    correct = predicted.eq(all_labels).sum().item()
    top1_acc = 100. * correct / len(all_labels)
    
    _, pred_top5 = all_preds.topk(5, 1, True, True)
    correct_5 = pred_top5.eq(all_labels.view(-1, 1).expand_as(pred_top5)).sum().item()
    top5_acc = 100. * correct_5 / len(all_labels)
    
    return running_loss / len(loader), top1_acc, top5_acc




### Main Funciton

In [2]:
def main():
    print("\n" + "="*70)
    print("OPTIMIZED TRAINING PIPELINE - BREAKING THROUGH 76% PLATEAU")
    print("="*70)
    print("\nKEY IMPROVEMENTS:")
    print("  ✓ Spatial attention on hands/face (critical joints)")
    print("  ✓ Multi-scale temporal pooling (captures patterns at different timescales)")
    print("  ✓ Late fusion (preserves stream-specific features)")
    print("  ✓ Deeper transformer blocks (better temporal modeling)")
    print("  ✓ Multi-scale pooling in classifier (avg + max)")
    print("="*70 + "\n")
    
    # Load data
    print("Loading JSON data...")
    with open(JSON_PATH, 'r') as f:
        json_data = json.load(f)
    
    train_ids, val_ids, test_ids = [], [], []
    labels_dict = {}
    
    unique_labels = sorted({info['action'][0] for info in json_data.values()})
    class_to_idx = {label: i for i, label in enumerate(unique_labels)}
    num_classes = len(unique_labels)
    
    print(f"Number of classes: {num_classes}")
    
    for video_id, info in json_data.items():
        file_path = os.path.join(FEATURES_DIR, f"{video_id}.npz")
        if os.path.exists(file_path):
            original_label = info['action'][0]
            labels_dict[video_id] = class_to_idx[original_label]
            
            if info['subset'] == 'train':
                train_ids.append(video_id)
            elif info['subset'] == 'val':
                val_ids.append(video_id)
            elif info['subset'] == 'test':
                test_ids.append(video_id)
    
    print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")
    
    # Create datasets
    print("\nCreating datasets...")
    augmentation = ModerateAugmentation()
    
    train_dataset = EnhancedSignLanguageDataset(
        train_ids, labels_dict, FEATURES_DIR, 
        seq_len=CONFIG['SEQ_LEN'], augmentation=augmentation, training=True
    )
    
    val_dataset = EnhancedSignLanguageDataset(
        val_ids, labels_dict, FEATURES_DIR, 
        seq_len=CONFIG['SEQ_LEN'], augmentation=None, training=False
    )
    
    test_dataset = EnhancedSignLanguageDataset(
        test_ids, labels_dict, FEATURES_DIR, 
        seq_len=CONFIG['SEQ_LEN'], augmentation=None, training=False
    )
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], 
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'], 
                            shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['BATCH_SIZE'], 
                             shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    
    # Create model
    print("\nInitializing OptimizedTTSTGCN model...")
    model = LightweightTTSTGCN(
        num_classes=num_classes, 
        base_ch=CONFIG['BASE_CHANNELS'],
        dropout=CONFIG['DROPOUT']
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Model size: {total_params * 4 / 1024 / 1024:.2f} MB")
    
    # Setup training
    criterion = LabelSmoothingCrossEntropy(smoothing=CONFIG['LABEL_SMOOTHING'])
    
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=CONFIG['LEARNING_RATE'], 
        weight_decay=CONFIG['WEIGHT_DECAY'],
        betas=(0.9, 0.999)
    )
    
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_epochs=CONFIG['WARMUP_EPOCHS'],
        max_epochs=CONFIG['EPOCHS'],
        min_lr=CONFIG['MIN_LR']
    )
    
    early_stopping = EarlyStopping(patience=CONFIG['PATIENCE'], min_delta=0.3)
    
    # Training loop
    print("\n" + "="*70)
    print("STARTING TRAINING")
    print("="*70)
    print("Expected milestones:")
    print("  Epoch 20:  Train ~35%, Val ~30%")
    print("  Epoch 50:  Train ~70%, Val ~65%")
    print("  Epoch 100: Train ~85%, Val ~78-80%")
    print("  Epoch 150: Train ~90%, Val ~80-83%")
    print("="*70 + "\n")
    
    best_val_acc = 0.0
    best_epoch = 0
    
    for epoch in range(CONFIG['EPOCHS']):
        start_time = time.time()
        
        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE, CONFIG
        )
        
        # Validate
        if (epoch + 1) % 10 == 0:
            val_loss, val_top1, val_top5 = validate(
                model, val_loader, criterion, DEVICE, num_tta=5
            )
        else:
            val_loss, val_top1, val_top5 = validate(
                model, val_loader, criterion, DEVICE, num_tta=1
            )
        
        scheduler.step()
        
        epoch_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        train_val_gap = train_acc - val_top1
        
        gap_indicator = "✓" if train_val_gap < 3 else ("⚠" if train_val_gap < 5 else "✗")
        
        print(f"Epoch {epoch+1:03d}/{CONFIG['EPOCHS']} | Time: {epoch_time:.1f}s | "
              f"LR: {current_lr:.6f} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Top-1: {val_top1:.2f}% | "
              f"Val Top-5: {val_top5:.2f}% | Gap: {train_val_gap:.2f}% {gap_indicator}")
        
        # Save best
        if val_top1 > best_val_acc:
            improvement = val_top1 - best_val_acc
            best_val_acc = val_top1
            best_epoch = epoch + 1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_top1,
                'config': CONFIG,
            }, 'best_optimized_model.pth')
            print(f"    💾 New best! Val: {best_val_acc:.2f}% (+{improvement:.2f}%)")
        
        # Milestone checks
        if epoch == 20 and val_top1 < 25:
            print("    ⚠️  WARNING: Low accuracy at epoch 20. Consider:")
            print("       → Reducing dropout to 0.2")
            print("       → Increasing learning rate to 0.0012")
        
        if epoch == 50 and val_top1 < 60:
            print("    ⚠️  WARNING: Slow progress at epoch 50.")
        
        if epoch > 100 and val_top1 > 78:
            print(f"    🎉 BREAKTHROUGH! Exceeded 78% plateau!")
        
        # Early stopping
        if early_stopping(val_top1):
            print(f"\n⏸ Early stopping at epoch {epoch+1}")
            print(f"   Best: {best_val_acc:.2f}% at epoch {best_epoch}")
            break
    
    # Final evaluation
    print("\n" + "="*70)
    print("FINAL EVALUATION ON TEST SET")
    print("="*70)
    
    checkpoint = torch.load('best_optimized_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    test_loss, test_top1, test_top5 = validate(
        model, test_loader, criterion, DEVICE, num_tta=10
    )
    
    print(f"\nFINAL RESULTS:")
    print(f"  Best Val Top-1:  {best_val_acc:.2f}% (Epoch {best_epoch})")
    print(f"  Test Top-1:      {test_top1:.2f}%")
    print(f"  Test Top-5:      {test_top5:.2f}%")
    
    improvement_over_baseline = best_val_acc - 76.67
    print(f"\nIMPROVEMENT OVER BASELINE (76.67%):")
    print(f"  Validation: +{improvement_over_baseline:.2f}%")
    
    if best_val_acc >= 82:
        print(f"\n🎉 EXCELLENT! Target exceeded (82%+)")
    elif best_val_acc >= 80:
        print(f"\n✓ GREAT! Reached 80%+ milestone")
    elif best_val_acc >= 78:
        print(f"\n✓ GOOD! Broke through 76% plateau")
    else:
        print(f"\n⚠️  Need further tuning")
    
    print(f"\n✓ Model saved: best_optimized_model.pth")
    
    # Generate predictions
    print("\nGenerating test predictions...")
    model.eval()
    all_preds = []
    all_video_ids = []
    
    with torch.no_grad():
        for idx in tqdm(range(len(test_dataset)), desc="Predicting"):
            joint, bone, _ = test_dataset[idx]
            joint = joint.unsqueeze(0).to(DEVICE)
            bone = bone.unsqueeze(0).to(DEVICE)
            
            tta_preds = []
            for _ in range(10):
                output = model(joint, bone)
                tta_preds.append(F.softmax(output, dim=1))
            
            pred = torch.stack(tta_preds).mean(0)
            pred_class = pred.argmax(1).item()
            
            all_preds.append(pred_class)
            all_video_ids.append(test_ids[idx])
    
    predictions_dict = {vid: int(pred) for vid, pred in zip(all_video_ids, all_preds)}
    with open('test_predictions_optimized.json', 'w') as f:
        json.dump(predictions_dict, f, indent=2)
    
    print("✓ Predictions saved: test_predictions_optimized.json")
    
    # Final recommendations
    print("\n" + "="*70)
    print("NEXT STEPS FOR EVEN HIGHER ACCURACY")
    print("="*70)
    
    if best_val_acc < 80:
        print("\nTO REACH 80%+:")
        print("  1. Train for more epochs (current best at {best_epoch})")
        print("  2. Try base_ch=36 (slightly more capacity)")
        print("  3. Reduce dropout to 0.2 if train-val gap < 3%")
        print("  4. Use ensemble of 2-3 models (+2-3%)")
    elif best_val_acc < 85:
        print("\nTO REACH 85%+:")
        print("  1. Ensemble 3 models with different seeds (+3-4%)")
        print("  2. Add more hand-specific augmentation")
        print("  3. Try focal loss for hard samples")
        print("  4. Use self-distillation (train student from teacher)")
    else:
        print("\n🎉 OUTSTANDING PERFORMANCE!")
        print("  Consider:")
        print("  1. Submitting to competition/benchmark")
        print("  2. Ensemble for final boost")
        print("  3. Document architecture for publication")
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE! 🎉")
    print("="*70)

In [3]:
if __name__ == "__main__":
    main()


OPTIMIZED TRAINING PIPELINE - BREAKING THROUGH 76% PLATEAU

KEY IMPROVEMENTS:
  ✓ Spatial attention on hands/face (critical joints)
  ✓ Multi-scale temporal pooling (captures patterns at different timescales)
  ✓ Late fusion (preserves stream-specific features)
  ✓ Deeper transformer blocks (better temporal modeling)
  ✓ Multi-scale pooling in classifier (avg + max)

Loading JSON data...
Number of classes: 300
Train: 3548, Val: 900, Test: 668

Creating datasets...
Train batches: 111, Val batches: 29

Initializing OptimizedTTSTGCN model...
Total parameters: 266,818
Model size: 1.02 MB

STARTING TRAINING
Expected milestones:
  Epoch 20:  Train ~35%, Val ~30%
  Epoch 50:  Train ~70%, Val ~65%
  Epoch 100: Train ~85%, Val ~78-80%
  Epoch 150: Train ~90%, Val ~80-83%



Epoch 001/300 | Time: 26.8s | LR: 0.000100 | Train Loss: 5.7129 | Train Acc: 0.34% | Val Loss: 5.7082 | Val Top-1: 0.22% | Val Top-5: 1.56% | Gap: 0.12% ✓
    💾 New best! Val: 0.22% (+0.22%)


Epoch 002/300 | Time: 17.6s | LR: 0.000200 | Train Loss: 5.7146 | Train Acc: 0.23% | Val Loss: 5.7028 | Val Top-1: 0.33% | Val Top-5: 1.89% | Gap: -0.11% ✓
    💾 New best! Val: 0.33% (+0.11%)


Epoch 003/300 | Time: 17.8s | LR: 0.000300 | Train Loss: 5.7042 | Train Acc: 0.56% | Val Loss: 5.6887 | Val Top-1: 0.56% | Val Top-5: 3.11% | Gap: 0.01% ✓
    💾 New best! Val: 0.56% (+0.22%)


Epoch 004/300 | Time: 18.1s | LR: 0.000400 | Train Loss: 5.6793 | Train Acc: 0.85% | Val Loss: 5.6368 | Val Top-1: 1.56% | Val Top-5: 4.56% | Gap: -0.71% ✓
    💾 New best! Val: 1.56% (+1.00%)


...

Epoch 288/300 | Time: 19.2s | LR: 0.000005 | Train Loss: 1.6028 | Train Acc: 78.72% | Val Loss: 2.0364 | Val Top-1: 70.67% | Val Top-5: 91.33% | Gap: 8.05% ✗


Epoch 289/300 | Time: 19.2s | LR: 0.000005 | Train Loss: 1.6709 | Train Acc: 69.93% | Val Loss: 2.0362 | Val Top-1: 70.56% | Val Top-5: 90.89% | Gap: -0.63% ✓

⏸ Early stopping at epoch 289
   Best: 71.22% at epoch 265

FINAL EVALUATION ON TEST SET



FINAL RESULTS:
  Best Val Top-1:  71.22% (Epoch 265)
  Test Top-1:      69.91%
  Test Top-5:      89.37%

IMPROVEMENT OVER BASELINE (76.67%):
  Validation: +-5.45%

⚠️  Need further tuning

✓ Model saved: best_optimized_model.pth

Generating test predictions...


Predicting: 100%|██████████| 668/668 [00:55<00:00, 12.03it/s]

✓ Predictions saved: test_predictions_optimized.json

NEXT STEPS FOR EVEN HIGHER ACCURACY

TO REACH 80%+:
  1. Train for more epochs (current best at {best_epoch})
  2. Try base_ch=36 (slightly more capacity)
  3. Reduce dropout to 0.2 if train-val gap < 3%
  4. Use ensemble of 2-3 models (+2-3%)

TRAINING COMPLETE! 🎉


### Test the Final Accuracy

In [5]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F

# Define the path to your best model file
BEST_MODEL_PATH = '/kaggle/working/best_optimized_model.pth'

# 1. Initialize the model structure (MUST use the same num_classes and base_ch as when saved)
# Assuming 'num_classes' and 'BASE_CHANNELS' from your CONFIG are available:
print("Initializing model structure...")
model = LightweightTTSTGCN(
    num_classes=300, 
    base_ch=CONFIG['BASE_CHANNELS'],
    dropout=CONFIG['DROPOUT']
).to(DEVICE)

# 2. Load the entire checkpoint dictionary
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

# 3. Load the model weights (state_dict is nested inside the checkpoint)
print(f"Loading weights from epoch {checkpoint['epoch']} (Acc: {checkpoint['val_acc']:.2f}%)")
model.load_state_dict(checkpoint['model_state_dict'])

# 4. Set the model to evaluation mode
model.eval()

Initializing model structure...
Loading weights from epoch 264 (Acc: 71.22%)


LightweightTTSTGCN(
  (bn_joint): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn_bone): BatchNorm2d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (joint_stream): Sequential(
    (0): LightweightSTGCNBlock(
      (gcn): AdaptiveGCN(
        (conv1): DepthwiseSeparableConv(
          (depthwise): Conv2d(4, 4, kernel_size=(1, 1), stride=(1, 1), groups=4)
          (pointwise): Conv2d(4, 24, kernel_size=(1, 1), stride=(1, 1))
          (bn): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiLU()
        )
        (conv2): DepthwiseSeparableConv(
          (depthwise): Conv2d(4, 4, kernel_size=(1, 1), stride=(1, 1), groups=4)
          (pointwise): Conv2d(4, 24, kernel_size=(1, 1), stride=(1, 1))
          (bn): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiLU()
        )
      )
      (tcn): DepthwiseSeparableConv(
        (depth

In [ ]:
# Load data
print("Loading JSON data...")
with open(JSON_PATH, 'r') as f:
    json_data = json.load(f)

train_ids, val_ids, test_ids = [], [], []
labels_dict = {}

unique_labels = sorted({info['action'][0] for info in json_data.values()})
class_to_idx = {label: i for i, label in enumerate(unique_labels)}
num_classes = len(unique_labels)

print(f"Number of classes: {num_classes}")

for video_id, info in json_data.items():
    file_path = os.path.join(FEATURES_DIR, f"{video_id}.npz")
    if os.path.exists(file_path):
        original_label = info['action'][0]
        labels_dict[video_id] = class_to_idx[original_label]
        
        if info['subset'] == 'train':
            train_ids.append(video_id)
        elif info['subset'] == 'val':
            val_ids.append(video_id)
        elif info['subset'] == 'test':
            test_ids.append(video_id)

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

# Create datasets
print("\nCreating datasets...")
augmentation = ModerateAugmentation()

train_dataset = EnhancedSignLanguageDataset(
    train_ids, labels_dict, FEATURES_DIR, 
    seq_len=CONFIG['SEQ_LEN'], augmentation=augmentation, training=True
)

val_dataset = EnhancedSignLanguageDataset(
    val_ids, labels_dict, FEATURES_DIR, 
    seq_len=CONFIG['SEQ_LEN'], augmentation=None, training=False
)

test_dataset = EnhancedSignLanguageDataset(
    test_ids, labels_dict, FEATURES_DIR, 
    seq_len=CONFIG['SEQ_LEN'], augmentation=None, training=False
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], 
                            shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'], 
                        shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['BATCH_SIZE'], 
                            shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Loading JSON data...
Number of classes: 300
Train: 3548, Val: 900, Test: 668

Creating datasets...
Train batches: 111, Val batches: 29


In [9]:
test_loader = DataLoader(test_dataset, batch_size=CONFIG['BATCH_SIZE'], 
                             shuffle=False, num_workers=2, pin_memory=True)
print("\nRunning inference and collecting predictions...")
all_preds_logits = []
all_labels = []
all_video_ids = []

with torch.no_grad():
    # Iterate through the test data
    for joint_batch, bone_batch, labels in tqdm(test_loader, desc="Predicting Test Data"):
        
        joint_batch = joint_batch.to(DEVICE)
        bone_batch = bone_batch.to(DEVICE)
        
        # --- TTA for final prediction consistency ---
        tta_preds = []
        for _ in range(10): # Using 10 TTA runs, matching your script's setting
            output = model(joint_batch, bone_batch)
            tta_preds.append(output)
            
        # Average the raw logits from all TTA runs
        avg_logits = torch.stack(tta_preds).mean(0)
        
        all_preds_logits.append(avg_logits.cpu())
        all_labels.append(labels.cpu())

# Concatenate all results
all_preds_logits = torch.cat(all_preds_logits, 0)
all_labels = torch.cat(all_labels, 0)

# Get the final predicted class index
_, final_predicted_indices = all_preds_logits.max(1)


Running inference and collecting predictions...


Predicting Test Data: 100%|██████████| 21/21 [00:09<00:00,  2.16it/s]


In [13]:
# Calculate Final Accuracy
correct_predictions = (final_predicted_indices == all_labels).sum().item()
test_top1_accuracy = 100. * correct_predictions / len(all_labels)

print("\n" + "=" * 50)
print(f"✅ Final Test Top-1 Accuracy: {test_top1_accuracy:.2f}%")
print("=" * 50)

# Optional: View a sample of predictions
idx_to_class = {i: label for label, i in class_to_idx.items()}
sample_count = 5 
print(f"\nSample Predictions (Top {sample_count}):")
for i in range(sample_count):
    true_label_index = all_labels[i].item()
    pred_label_index = final_predicted_indices[i].item()
    
    true_class = idx_to_class.get(true_label_index, f"Index {true_label_index}")
    pred_class = idx_to_class.get(pred_label_index, f"Index {pred_label_index}")
    
    print(f"  Sample {i+1} | True Class: {true_class} | Predicted Class: {pred_class}" )


✅ Final Test Top-1 Accuracy: 69.91%

Sample Predictions (Top 5):
  Sample 1 | True Class: 247 | Predicted Class: 247
  Sample 2 | True Class: 46 | Predicted Class: 46
  Sample 3 | True Class: 46 | Predicted Class: 46
  Sample 4 | True Class: 229 | Predicted Class: 234
  Sample 5 | True Class: 252 | Predicted Class: 252
